# VoiceOfBank — 06 Topic Modelling
**Notebook 6 of 7** — Run on Google Colab (GPU required)

Extract complaint topics from 1-2 star reviews using LDA and BERTopic.

**What we are answering:**
- What are customers actually complaining about?
- Do complaint topics differ between challenger and traditional banks?
- Which topics are associated with the Feb-Apr 2026 spike in Barclays/Lloyds?

**Two approaches:**
- LDA (Latent Dirichlet Allocation) — classical probabilistic topic model, runs on CPU
- BERTopic — neural topic model using sentence transformers, GPU accelerated

**Input:** `data/processed/reviews_bert.csv` (from Google Drive)

**Output:** `data/processed/topics_lda.csv` · `data/processed/topics_bert.csv`


## 1. Install Packages

In [ ]:
import subprocess
subprocess.run(
    ['pip', 'install', 'bertopic', 'sentence-transformers',
     'umap-learn', 'hdbscan', '--quiet'],
    check=True
)
print('Packages ready.')


## 2. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
from pathlib import Path
from collections import Counter

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
COLORS     = sns.color_palette('muted')
BANKS      = ['Monzo','Starling','Barclays','HSBC','NatWest','Lloyds']
CHALLENGER = ['Monzo','Starling']
PALETTE    = dict(zip(BANKS, sns.color_palette('tab10', 6)))

print('Setup complete.')


## 3. Mount Drive and Load Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/VoiceOfBank/data/processed/'

df = pd.read_csv(DRIVE_PATH + 'reviews_bert.csv', parse_dates=['date'])

# Complaint reviews: 1-2 star only
complaints = df[df['is_complaint']==1].reset_index(drop=True)

print(f'Total reviews    : {len(df):,}')
print(f'Complaint reviews: {len(complaints):,}  ({len(complaints)/len(df)*100:.1f}%)')
print()
print('Complaints per bank:')
print(complaints['bank'].value_counts().to_string())
print()
print('Date range:', complaints['date'].min().date(),
      '->', complaints['date'].max().date())


## 4. LDA Topic Modelling

LDA is a probabilistic model that represents each document as a mixture of topics
and each topic as a mixture of words. It is fast, interpretable, and works well
as a baseline before the more powerful BERTopic.

We run LDA on the cleaned text from all complaint reviews combined,
then inspect the top words per topic to assign human-readable labels.


### 4.1 Fit LDA

In [ ]:
N_TOPICS = 8  # number of complaint topics

vectorizer = CountVectorizer(
    max_features = 5000,
    ngram_range  = (1, 2),
    min_df       = 3,
    max_df       = 0.90,
)

X_counts = vectorizer.fit_transform(complaints['text_clean'].fillna(''))
vocab    = vectorizer.get_feature_names_out()

print(f'Vocabulary size : {len(vocab):,}')
print(f'Matrix shape    : {X_counts.shape}')
print()
print(f'Fitting LDA with {N_TOPICS} topics...')

lda = LatentDirichletAllocation(
    n_components  = N_TOPICS,
    max_iter      = 20,
    learning_method = 'batch',
    random_state  = 42,
    n_jobs        = -1,
)
lda.fit(X_counts)
print('LDA fitted.')


### 4.2 Inspect Topics

In [ ]:
def print_lda_topics(model, vocab, n_words=12):
    for topic_idx, topic in enumerate(model.components_):
        top_words = [vocab[i] for i in topic.argsort()[::-1][:n_words]]
        print(f'  Topic {topic_idx}: {" | ".join(top_words)}')

print('LDA Topics — top 12 words each:')
print_lda_topics(lda, vocab)
print()
print('Assign labels based on the words above.')
print('Edit TOPIC_LABELS below to match what you see.')


In [ ]:
# Assign human-readable labels to each LDA topic
# Edit these after inspecting the top words above
TOPIC_LABELS = {
    0: 'App Performance',
    1: 'Account Access',
    2: 'Customer Service',
    3: 'Payments & Transfers',
    4: 'Card Issues',
    5: 'Fraud & Security',
    6: 'Account Closure',
    7: 'Fees & Charges',
}

# Assign dominant topic to each complaint
topic_dist       = lda.transform(X_counts)
dominant_topic   = topic_dist.argmax(axis=1)
complaints       = complaints.copy()
complaints['lda_topic']       = dominant_topic
complaints['lda_topic_label'] = complaints['lda_topic'].map(TOPIC_LABELS)
complaints['lda_topic_prob']  = topic_dist.max(axis=1)

print('LDA topic distribution (all complaints):')
topic_counts = complaints['lda_topic_label'].value_counts()
for topic, count in topic_counts.items():
    pct = count / len(complaints) * 100
    print(f'  {topic:<25}: {count:>4}  ({pct:.1f}%)')


### 4.3 LDA Topics by Bank

In [ ]:
topic_bank = complaints.groupby(['bank','lda_topic_label']).size().unstack(fill_value=0)
topic_bank_pct = topic_bank.div(topic_bank.sum(axis=1), axis=0).mul(100).round(1)
topic_bank_pct = topic_bank_pct[sorted(topic_bank_pct.columns)]

fig, ax = plt.subplots(figsize=(13, 6))
topic_bank_pct.plot(kind='bar', ax=ax, colormap='tab10', edgecolor='white', width=0.8)
ax.set_title('LDA Complaint Topics by Bank (%)', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('% of Complaints')
ax.set_xticklabels(ax.get_xticklabels(), rotation=15)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
sns.despine()
plt.tight_layout()
plt.show()

print('Topic distribution per bank:')
print(topic_bank_pct.to_string())


## 5. BERTopic

BERTopic uses sentence transformers to embed documents into a dense vector space,
then clusters them with HDBSCAN and extracts topic keywords using TF-IDF.

Advantages over LDA:
- Understands semantic meaning, not just word co-occurrence
- Automatically determines the number of topics
- Each topic has a coherent narrative rather than a bag of words
- Topic -1 is the outlier topic (reviews that do not fit any cluster)


### 5.1 Fit BERTopic

In [ ]:
print('Loading sentence transformer...')
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# UMAP for dimensionality reduction before clustering
umap_model = UMAP(
    n_neighbors = 15,
    n_components = 5,
    min_dist    = 0.0,
    metric      = 'cosine',
    random_state = 42,
)

# HDBSCAN for density-based clustering
hdbscan_model = HDBSCAN(
    min_cluster_size = 50,
    min_samples      = 10,
    metric           = 'euclidean',
    cluster_selection_method = 'eom',
    prediction_data  = True,
)

topic_model = BERTopic(
    embedding_model = embedding_model,
    umap_model      = umap_model,
    hdbscan_model   = hdbscan_model,
    nr_topics       = 'auto',
    top_n_words     = 10,
    verbose         = True,
)

print('Fitting BERTopic on complaint reviews...')
print(f'Input: {len(complaints):,} complaint reviews')

docs   = complaints['text_clean'].fillna('').tolist()
topics, probs = topic_model.fit_transform(docs)

complaints = complaints.copy()
complaints['bert_topic'] = topics

topic_info = topic_model.get_topic_info()
print(f'\nTopics found: {len(topic_info)-1}  (excluding outlier topic -1)')
print(f'Outliers    : {(np.array(topics)==-1).sum():,}  reviews')


### 5.2 Inspect BERTopic Topics

In [ ]:
print('Top topics by size:')
print(topic_info[topic_info['Topic']!=-1].head(15)[['Topic','Count','Name']].to_string(index=False))
print()
print('Top words per topic:')
for _, row in topic_info[topic_info['Topic']!=-1].head(10).iterrows():
    t = row['Topic']
    words = [w for w, _ in topic_model.get_topic(t)[:8]]
    print(f'  Topic {t:>3} ({row["Count"]:>4} docs): {" | ".join(words)}')


In [ ]:
# Assign human-readable labels based on the words above
# Edit this dict after inspecting the topic words
BERT_TOPIC_LABELS = {
    -1 : 'Outlier',
     0 : 'App Crashes',
     1 : 'Account Blocked',
     2 : 'Customer Service',
     3 : 'Payment Failed',
     4 : 'Card Not Working',
     5 : 'Fraud & Scam',
     6 : 'Login Issues',
     7 : 'Transfer Delay',
}

# Map topic numbers to labels
# Fill any topic not in dict with 'Other'
complaints['bert_topic_label'] = complaints['bert_topic'].map(
    lambda x: BERT_TOPIC_LABELS.get(x, f'Topic {x}')
)

print('BERTopic distribution (excluding outliers):')
non_outlier = complaints[complaints['bert_topic']!=-1]
topic_counts = non_outlier['bert_topic_label'].value_counts()
for topic, count in topic_counts.items():
    pct = count / len(non_outlier) * 100
    print(f'  {topic:<25}: {count:>4}  ({pct:.1f}%)')


### 5.3 BERTopic Heatmap — Topics by Bank

In [ ]:
# Topic distribution per bank — heatmap
non_outlier = complaints[complaints['bert_topic']!=-1].copy()

heatmap_data = non_outlier.groupby(['bank','bert_topic_label']).size().unstack(fill_value=0)
heatmap_pct  = heatmap_data.div(heatmap_data.sum(axis=1), axis=0).mul(100).round(1)

fig, ax = plt.subplots(figsize=(13, 6))
sns.heatmap(
    heatmap_pct,
    annot  = True,
    fmt    = '.1f',
    cmap   = 'YlOrRd',
    linewidths = 0.5,
    ax     = ax,
)
ax.set_title('BERTopic Complaint Topics by Bank (% of complaints)',
             fontweight='bold')
ax.set_ylabel('')
ax.set_xlabel('Complaint Topic')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()


### 5.4 Topic Spike Analysis — Feb-Apr 2026

In [ ]:
# Investigate the complaint spike for Barclays and Lloyds in Feb-Apr 2026
spike_banks  = ['Barclays','Lloyds']
spike_start  = '2026-02-01'
spike_end    = '2026-04-30'

spike = complaints[
    (complaints['bank'].isin(spike_banks)) &
    (complaints['date'] >= spike_start) &
    (complaints['date'] <= spike_end) &
    (complaints['bert_topic'] != -1)
]

pre_spike = complaints[
    (complaints['bank'].isin(spike_banks)) &
    (complaints['date'] < spike_start) &
    (complaints['bert_topic'] != -1)
]

print(f'Complaints during spike (Feb-Apr 2026): {len(spike):,}')
print(f'Complaints before spike               : {len(pre_spike):,}')
print()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, (label, data) in enumerate([
    ('Before Spike', pre_spike),
    ('During Spike (Feb-Apr 2026)', spike)
]):
    counts = data['bert_topic_label'].value_counts().head(8)
    axes[i].barh(counts.index[::-1], counts.values[::-1],
                 color='#ef4444' if i==1 else '#4f8ef7', edgecolor='white')
    axes[i].set_title(f'Barclays + Lloyds Complaints\n{label}',
                      fontweight='bold')
    axes[i].set_xlabel('Number of Complaints')

sns.despine()
plt.tight_layout()
plt.show()

print('What changed during the spike?')
print('Compare the dominant topics before vs during to identify the incident.')


## 6. Complaint Trend Over Time

In [ ]:
# Monthly complaint volume by bank
complaints['year_month_dt'] = pd.to_datetime(complaints['year_month'])
monthly_complaints = complaints.groupby(
    ['year_month_dt','bank']
).size().reset_index(name='count')

fig, ax = plt.subplots(figsize=(14, 5))
for bank in BANKS:
    sub = monthly_complaints[monthly_complaints['bank']==bank]
    lw  = 2.5 if bank in CHALLENGER else 1.5
    ls  = '-'  if bank in CHALLENGER else '--'
    ax.plot(sub['year_month_dt'], sub['count'],
            label=bank, color=PALETTE[bank],
            linewidth=lw, linestyle=ls, marker='o', markersize=3)

ax.set_xlabel('Month')
ax.set_ylabel('Number of 1-2 Star Reviews')
ax.set_title('Monthly Complaint Volume by Bank', fontweight='bold')
ax.legend(ncol=3, fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=30)
sns.despine()
plt.tight_layout()
plt.show()


## 7. Save

In [ ]:
# Save LDA results
lda_out = complaints[[
    'reviewId','bank','date','text','rating','sentiment',
    'bert_label','lda_topic','lda_topic_label','lda_topic_prob'
]].copy()
lda_out.to_csv(DRIVE_PATH + 'topics_lda.csv', index=False)
print('Saved: topics_lda.csv')

# Save BERTopic results
bert_out = complaints[[
    'reviewId','bank','date','text','rating','sentiment',
    'bert_label','bert_topic','bert_topic_label'
]].copy()
bert_out.to_csv(DRIVE_PATH + 'topics_bert.csv', index=False)
print('Saved: topics_bert.csv')
print()
print('Download both files and save to VoiceOfBank/data/processed/')
print()
print('Next: 07_classification.ipynb  (run on Colab)')
